# Airport Selection and Analysis

Choosing the right base of operations is a critical logistics decision for airborne
campaigns. The airport must have adequate runway length and surface for the aircraft,
be close enough to the study area to maximize on-station time, and have the
infrastructure to support flight operations. This notebook demonstrates HyPlan's
`airports` module for finding, filtering, and analyzing airports.

We cover:

1. Initializing the airport database with filters (country, runway, surface type)
2. Finding the nearest airport to a study site
3. Finding multiple nearby airports
4. Searching within a radius
5. Inspecting airport details with the `Airport` class
6. Analyzing runway information
7. Comparing candidate airports
8. Exporting airports to GeoJSON
9. Visualizing airports on a map

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from hyplan.airports import (
    Airport,
    initialize_data,
    find_nearest_airport,
    find_nearest_airports,
    airports_within_radius,
    get_airports,
    get_runways,
    get_airport_details,
    get_longest_runway,
    get_runway_details,
    generate_geojson,
)

## 1. Initializing the Airport Database

`initialize_data()` downloads and caches airport/runway data from the OurAirports
database. You can filter by:
- **countries** — ISO country codes (e.g., "US", "CA")
- **min_runway_length** — Minimum runway length in feet
- **runway_surface** — Accepted surface types (e.g., asphalt, concrete)
- **airport_types** — Airport categories (e.g., "large_airport", "medium_airport")

In [2]:
# Initialize with filters suitable for a King Air B200
# (needs ~5000 ft paved runway)
initialize_data(
    countries=["US"],
    min_runway_length=5000,  # feet
    runway_surface=["ASPH", "CONC", "ASP", "CON", "MAC", "BIT", "Asphalt", "Concrete"],
)

# Check how many airports match
gdf_airports = get_airports()
print(f"Airports matching filters: {len(gdf_airports)}")

Airports matching filters: 1798


## 2. Finding the Nearest Airport

`find_nearest_airport()` returns the ICAO code of the closest airport to a given
latitude/longitude. This is a quick way to identify the most convenient base.

In [3]:
# Study site: Central Valley, California
study_lat, study_lon = 36.5, -119.5

nearest = find_nearest_airport(study_lat, study_lon)
print(f"Nearest airport to ({study_lat}, {study_lon}): {nearest}")

# Get details
apt = Airport(nearest)
print(f"  Name: {apt.name}")
print(f"  IATA: {apt.iata_code}")
print(f"  Location: ({apt.latitude:.4f}, {apt.longitude:.4f})")
print(f"  Elevation: {apt.elevation}")
print(f"  Municipality: {apt.municipality}")

Nearest airport to (36.5, -119.5): KVIS
  Name: Visalia Municipal Airport
  IATA: VIS
  Location: (36.3187, -119.3930)
  Elevation: 89.91599999999998 meter
  Municipality: Visalia


## 3. Finding Multiple Nearby Airports

`find_nearest_airports()` returns the N closest airports, giving you a ranked
list of candidates to evaluate.

In [4]:
# Find the 5 nearest airports
top5 = find_nearest_airports(study_lat, study_lon, n=5)
print(f"Top 5 nearest airports to ({study_lat}, {study_lon}):")
for icao in top5:
    a = Airport(icao)
    print(f"  {icao} — {a.name} ({a.municipality})")

Top 5 nearest airports to (36.5, -119.5):
  KVIS — Visalia Municipal Airport (Visalia)
  KHJO — Hanford Municipal Airport (Hanford)
  KFAT — Fresno Yosemite International Airport (Fresno)
  24CL — Boswell Airport (Corcoran)
  KNLC — Lemoore Naval Air Station (Reeves Field) Airport (Lemoore)


## 4. Searching Within a Radius

`airports_within_radius()` finds all airports within a specified distance.
Use `return_details=True` to get a full GeoDataFrame with airport metadata.

In [5]:
# ICAO codes only
within_50mi = airports_within_radius(study_lat, study_lon, radius=50, unit="miles")
print(f"Airports within 50 miles: {len(within_50mi)}")
print(f"  ICAO codes: {within_50mi}")

# With full details as a GeoDataFrame
within_100km = airports_within_radius(
    study_lat, study_lon, radius=100, unit="kilometers", return_details=True
)
print(f"\nAirports within 100 km: {len(within_100km)}")
within_100km

Airports within 50 miles: 6
  ICAO codes: ['24CL', 'KFAT', 'KHJO', 'KNLC', 'KPTV', 'KVIS']

Airports within 100 km: 9


,type,name,latitude,longitude,elevation_ft,continent,iso_country,iso_region,municipality,icao_code,iata_code,geometry,distance_m
ident,,,,,,,,,,,,,
KVIS,medium_airport,Visalia Municipal Airport,36.318699,-119.392998,295.0,NaN,US,US-CA,Visalia,KVIS,VIS,POINT (-119.393 36.3187),22318.295110
KHJO,small_airport,Hanford Municipal Airport,36.316700,-119.627998,240.0,NaN,US,US-CA,Hanford,KHJO,NaN,POINT (-119.628 36.3167),23380.241474
KFAT,large_airport,Fresno Yosemite International Airport,36.775767,-119.718018,336.0,NaN,US,US-CA,Fresno,KFAT,FAT,POINT (-119.71802 36.77577),36313.649499
KNLC,medium_airport,Lemoore Naval Air Station (Reeves Field) Airport,36.333000,-119.952004,232.0,NaN,US,US-CA,Lemoore,KNLC,NLC,POINT (-119.952 36.333),44504.852208
24CL,small_airport,Boswell Airport,36.088444,-119.541392,205.0,NaN,US,US-CA,Corcoran,24CL,NaN,POINT (-119.54139 36.08844),45913.045702
KPTV,small_airport,Porterville Municipal Airport,36.029598,-119.063004,442.0,NaN,US,US-CA,Porterville,KPTV,PTV,POINT (-119.063 36.0296),65352.349921
KMAE,small_airport,Madera Municipal Airport,36.988602,-120.112000,255.0,NaN,US,US-CA,Madera,KMAE,MAE,POINT (-120.112 36.9886),76975.674961
KC80,small_airport,New Coalinga Municipal Airport,36.163101,-120.293999,622.0,NaN,US,US-CA,Coalinga,KC80,CLG,POINT (-120.294 36.1631),80387.412944
KDLO,small_airport,Delano Municipal Airport,35.745602,-119.237000,314.0,NaN,US,US-CA,Delano,KDLO,NaN,POINT (-119.237 35.7456),87147.736645


## 5. The Airport Class

The `Airport` class provides a convenient object-oriented interface for inspecting
a single airport's properties: name, codes, location, elevation, and runway data.

In [6]:
# Inspect a well-known airport
lax = Airport("KLAX")

print(f"Name:         {lax.name}")
print(f"ICAO:         {lax.icao_code}")
print(f"IATA:         {lax.iata_code}")
print(f"Country:      {lax.country}")
print(f"Municipality: {lax.municipality}")
print(f"Location:     ({lax.latitude:.4f}, {lax.longitude:.4f})")
print(f"Elevation:    {lax.elevation}")
print(f"Elevation ft: {lax.elevation_ft} ft")
print(f"Geometry:     {lax.geometry}")
print(f"\nRunways:")
lax.runways

Name:         Los Angeles International Airport
ICAO:         KLAX
IATA:         LAX
Country:      US
Municipality: Los Angeles
Location:     (33.9425, -118.4080)
Elevation:    38.099999999999994 meter
Elevation ft: 125.0 ft
Geometry:     POINT (-118.407997 33.942501)

Runways:


,airport_ident,length_ft,width_ft,surface,le_heading_degT,he_heading_degT
23445,KLAX,8926.0,150.0,CON,83.0,263.0
23446,KLAX,10885.0,150.0,CON,83.0,263.0
23447,KLAX,12923.0,150.0,CON,83.0,263.0
23448,KLAX,11095.0,200.0,CON,83.0,263.0


## 6. Runway Analysis

Use `get_runway_details()` and `get_longest_runway()` to evaluate whether
candidate airports can accommodate your aircraft.

In [7]:
# Runway details for several candidate airports
candidates = ["KLAX", "KBFL", "KFAT", "KVIS"]
runway_df = get_runway_details(candidates)
print(f"Runway details for {len(candidates)} airports:")
runway_df

Runway details for 4 airports:


,airport_ident,length_ft,width_ft,surface,le_heading_degT,he_heading_degT
21398,KBFL,10849.0,150.0,ASP,135.4,315.5
21399,KBFL,7700.0,100.0,ASP,135.5,315.5
22488,KFAT,9539.0,150.0,ASP,125.4,305.4
22489,KFAT,8008.0,150.0,ASP,125.0,305.0
23445,KLAX,8926.0,150.0,CON,83.0,263.0
23446,KLAX,10885.0,150.0,CON,83.0,263.0
23447,KLAX,12923.0,150.0,CON,83.0,263.0
23448,KLAX,11095.0,200.0,CON,83.0,263.0
25782,KVIS,6562.0,150.0,ASP,134.9,314.9


In [8]:
# Check longest runway at each candidate
print("Longest runway at each airport:")
for icao in candidates:
    longest = get_longest_runway(icao)
    a = Airport(icao)
    print(f"  {icao} ({a.name}): {longest:.0f} ft")

Longest runway at each airport:
  KLAX (Los Angeles International Airport): 12923 ft
  KBFL (Meadows Field): 10849 ft
  KFAT (Fresno Yosemite International Airport): 9539 ft
  KVIS (Visalia Municipal Airport): 6562 ft


## 7. Comparing Candidate Airports

Build a comparison table to evaluate airports on multiple criteria: distance to
study site, runway length, and elevation.

In [9]:
from hyplan.geometry import haversine

# Compare the top 5 nearest airports
rows = []
for icao in top5:
    a = Airport(icao)
    dist_km = haversine(study_lat, study_lon, a.latitude, a.longitude) / 1000
    longest_rwy = get_longest_runway(icao)
    rows.append({
        "ICAO": icao,
        "Name": a.name,
        "Municipality": a.municipality,
        "Distance (km)": f"{dist_km:.1f}",
        "Longest Runway (ft)": f"{longest_rwy:.0f}",
        "Elevation (ft)": f"{a.elevation_ft:.0f}" if a.elevation_ft else "N/A",
    })

comparison = pd.DataFrame(rows)
comparison

,ICAO,Name,Municipality,Distance (km),Longest Runway (ft),Elevation (ft)
0,KVIS,Visalia Municipal Airport,Visalia,22.3,6562,295
1,KHJO,Hanford Municipal Airport,Hanford,23.4,5179,240
2,KFAT,Fresno Yosemite International Airport,Fresno,36.3,9539,336
3,24CL,Boswell Airport,Corcoran,45.9,6815,205
4,KNLC,Lemoore Naval Air Station (Reeves Field) Airport,Lemoore,44.5,13502,232


## 8. Batch Airport Details

`get_airport_details()` retrieves metadata for multiple airports at once, and
`get_airports()` returns the full filtered airport GeoDataFrame.

In [10]:
# Batch details
details = get_airport_details(top5)
details

,type,name,latitude,longitude,elevation_ft,continent,iso_country,iso_region,municipality,icao_code,iata_code,geometry
ident,,,,,,,,,,,,
24CL,small_airport,Boswell Airport,36.088444,-119.541392,205.0,NaN,US,US-CA,Corcoran,24CL,NaN,POINT (-119.54139 36.08844)
KFAT,large_airport,Fresno Yosemite International Airport,36.775767,-119.718018,336.0,NaN,US,US-CA,Fresno,KFAT,FAT,POINT (-119.71802 36.77577)
KHJO,small_airport,Hanford Municipal Airport,36.316700,-119.627998,240.0,NaN,US,US-CA,Hanford,KHJO,NaN,POINT (-119.628 36.3167)
KNLC,medium_airport,Lemoore Naval Air Station (Reeves Field) Airport,36.333000,-119.952004,232.0,NaN,US,US-CA,Lemoore,KNLC,NLC,POINT (-119.952 36.333)
KVIS,medium_airport,Visalia Municipal Airport,36.318699,-119.392998,295.0,NaN,US,US-CA,Visalia,KVIS,VIS,POINT (-119.393 36.3187)


## 9. Exporting to GeoJSON

`generate_geojson()` creates a GeoJSON file of airports for use in GIS tools
or interactive maps. You can export all airports or a specific subset.

In [11]:
# Export candidate airports to GeoJSON (commented to avoid writing files)
# generate_geojson("candidate_airports.geojson", icao_codes=top5)
# generate_geojson("all_filtered_airports.geojson")
print("Export examples:")
print('  generate_geojson("candidate_airports.geojson", icao_codes=top5)')
print('  generate_geojson("all_filtered_airports.geojson")')

Export examples:
  generate_geojson("candidate_airports.geojson", icao_codes=top5)
  generate_geojson("all_filtered_airports.geojson")


## 10. Interactive Airport Map

Use [Folium](https://python-visualization.github.io/folium/) to visualize airports
on an interactive map with satellite imagery, clickable markers, and a search
radius circle.

In [12]:
import folium

m = folium.Map(location=[study_lat, study_lon], zoom_start=9,
               tiles="CartoDB positron")

# Study site marker
folium.Marker(
    [study_lat, study_lon],
    popup="Study Site",
    icon=folium.Icon(color="red", icon="star", prefix="fa"),
).add_to(m)

# 100 km search radius
folium.Circle(
    [study_lat, study_lon],
    radius=100_000,  # meters
    color="red", fill=False, dash_array="10",
    popup="100 km radius",
).add_to(m)

# All airports in the region (small dots)
for _, row in gdf_airports.iterrows():
    folium.CircleMarker(
        [row["latitude"], row["longitude"]],
        radius=3, color="gray", fill=True, fill_opacity=0.4,
    ).add_to(m)

# Candidate airports (larger markers with popups)
colors = ["blue", "green", "purple", "orange", "darkred"]
for icao, color in zip(top5, colors):
    a = Airport(icao)
    dist_km = haversine(study_lat, study_lon, a.latitude, a.longitude) / 1000
    rwy = get_longest_runway(icao)
    popup_html = (
        f"<b>{icao}</b> — {a.name}<br>"
        f"Municipality: {a.municipality}<br>"
        f"Distance: {dist_km:.1f} km<br>"
        f"Longest runway: {rwy:.0f} ft<br>"
        f"Elevation: {a.elevation_ft:.0f} ft"
    )
    folium.Marker(
        [a.latitude, a.longitude],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=icao,
        icon=folium.Icon(color=color, icon="plane", prefix="fa"),
    ).add_to(m)

m

## 11. Re-initializing with Different Filters

You can call `initialize_data()` again with different criteria. For example,
when planning for a larger aircraft like the NASA ER-2, you need longer runways.

In [13]:
# Re-initialize for NASA ER-2 (needs 8000+ ft paved runway)
initialize_data(
    countries=["US"],
    min_runway_length=8000,
    runway_surface=["ASPH", "CONC", "ASP", "CON", "Asphalt", "Concrete"],
)

gdf_er2 = get_airports()
print(f"Airports suitable for ER-2: {len(gdf_er2)}")

# Find candidates near the same study site
er2_candidates = airports_within_radius(study_lat, study_lon, radius=150, unit="miles")
print(f"\nER-2 capable airports within 150 miles:")
for icao in er2_candidates:
    a = Airport(icao)
    dist = haversine(study_lat, study_lon, a.latitude, a.longitude) / 1000
    rwy = get_longest_runway(icao)
    print(f"  {icao} — {a.name}: {rwy:.0f} ft runway, {dist:.0f} km away")

Airports suitable for ER-2: 352

ER-2 capable airports within 150 miles:
  KBFL — Meadows Field: 10849 ft runway, 125 km away
  KFAT — Fresno Yosemite International Airport: 9539 ft runway, 36 km away
  KNLC — Lemoore Naval Air Station (Reeves Field) Airport: 13502 ft runway, 45 km away
  KSMX — Santa Maria Public Airport Captain G Allan Hancock Field: 8004 ft runway, 198 km away
  KVBG — Vandenberg Space Force Base: 15000 ft runway, 219 km away


## Summary

| Function/Class | Purpose |
|----------------|---------|
| `initialize_data(countries, min_runway_length, ...)` | Load and filter the airport database |
| `find_nearest_airport(lat, lon)` | ICAO code of the closest airport |
| `find_nearest_airports(lat, lon, n)` | N closest airports ranked by distance |
| `airports_within_radius(lat, lon, radius, unit)` | All airports within a distance |
| `Airport(icao)` | Object with name, location, elevation, runways |
| `get_airport_details(icao_list)` | Batch metadata lookup |
| `get_runway_details(icao_list)` | Runway specifications for multiple airports |
| `get_longest_runway(icao)` | Longest runway length in feet |
| `get_airports()` | Full filtered airport GeoDataFrame |
| `get_runways()` | Full filtered runway DataFrame |
| `generate_geojson(filepath, icao_codes)` | Export airports to GeoJSON |